# Lifestyle Planner - Rich Planning Engine

This notebook documents the upgraded lifestyle planner. It uses a realistic synthetic cohort because a complete labelled dataset for calories, protein, constraints, schedule, diet pattern, sleep, stress, and monthly planning is not available in the project. The synthetic targets are generated from Mifflin-St Jeor, activity multipliers, conservative goal deltas, and public-health guardrails.

Planning references used by the app:

- CDC adult activity guidance: https://www.cdc.gov/physical-activity-basics/guidelines/adults.html
- Physical Activity Guidelines for Americans PDF: https://www.cdc.gov/physical-activity/media/pdfs/Physical_Activity_Guidelines_2nd_edition.pdf
- CDC adult sleep guidance: https://www.cdc.gov/sleep/about/index.html
- Dietary Guidelines for Americans 2020-2025: https://www.dietaryguidelines.gov/sites/default/files/2020-12/Dietary_Guidelines_for_Americans_2020-2025.pdf

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.insert(0, str(Path('..', 'scripts').resolve()))
from train_lifestyle import (
    FEATURES, TARGETS, CATEGORICAL_FEATURES, NUMERIC_FEATURES,
    synth_dataset, make_preprocessor, candidate_models, tune_model
)

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

ROOT = Path('..').resolve()
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

## 1. Build the planning dataset

In [ ]:
df = synth_dataset(n=9000, seed=42)
print('Shape:', df.shape)
display(df.head())
display(df.describe().T[['mean','std','min','max']].round(2))

## 2. EDA: who is in the cohort?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.histplot(df, x='age', bins=24, ax=axes[0,0]); axes[0,0].set_title('Age')
sns.histplot(df, x='bmi', bins=24, ax=axes[0,1]); axes[0,1].set_title('BMI')
sns.histplot(df, x='sleep_hours', bins=20, ax=axes[0,2]); axes[0,2].set_title('Sleep')
sns.countplot(df, x='goal', ax=axes[1,0]); axes[1,0].tick_params(axis='x', rotation=35)
sns.countplot(df, x='activity', ax=axes[1,1]); axes[1,1].tick_params(axis='x', rotation=35)
sns.countplot(df, x='diet_type', ax=axes[1,2]); axes[1,2].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

## 3. EDA: target behaviour

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(df, x='goal', y='calories', ax=axes[0]); axes[0].tick_params(axis='x', rotation=35)
sns.boxplot(df, x='activity', y='calories', ax=axes[1]); axes[1].tick_params(axis='x', rotation=35)
sns.scatterplot(df.sample(1800, random_state=42), x='weight_kg', y='protein_g', hue='goal', alpha=.45, ax=axes[2])
plt.tight_layout()
plt.show()

display(df.groupby(['goal','activity'])[['calories','protein_g','workout_days','steps_now']].median().round(1).head(20))

## 4. Train/test split and evaluator

In [ ]:
X = df[FEATURES]
y = df[TARGETS]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Train/test:', X_train.shape, X_test.shape)

def evaluate_model(name, estimator):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring={'mae':'neg_mean_absolute_error', 'r2':'r2'}, n_jobs=-1)
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    mae = np.mean(np.abs(y_test.to_numpy() - pred), axis=0)
    row = {
        'model': name,
        'cv_mae': -scores['test_mae'].mean(),
        'cv_r2': scores['test_r2'].mean(),
        'mae_calories': mae[0],
        'mae_protein': mae[1],
        'r2_calories': r2_score(y_test.iloc[:,0], pred[:,0]),
        'r2_protein': r2_score(y_test.iloc[:,1], pred[:,1]),
        'estimator': estimator,
    }
    print(f"{name:28s} cv_mae={row['cv_mae']:.1f} test_mae_kcal={row['mae_calories']:.1f} test_mae_protein={row['mae_protein']:.1f}")
    return row

## 5. Baseline trials

In [ ]:
results = [evaluate_model(name, model) for name, model in candidate_models().items()]
leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values('mae_calories')
display(leaderboard.style.format('{:.3f}', subset=['cv_mae','cv_r2','mae_calories','mae_protein','r2_calories','r2_protein']))

## 6. Fine tuning

In [ ]:
tuned_name, tuned_model, tuned_params = tune_model(X_train, y_train)
print(tuned_params)
results.append(evaluate_model(tuned_name, tuned_model))
leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values(['mae_calories','mae_protein'])
display(leaderboard.style.format('{:.3f}', subset=['cv_mae','cv_r2','mae_calories','mae_protein','r2_calories','r2_protein']))

## 7. Inspect residuals

In [ ]:
best_row = leaderboard.iloc[0]
best = next(r for r in results if r['model'] == best_row['model'])
best_model = best['estimator']
pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test['calories'], pred[:,0], alpha=.25)
axes[0].plot([y_test['calories'].min(), y_test['calories'].max()], [y_test['calories'].min(), y_test['calories'].max()], 'k--')
axes[0].set_title('Calories: actual vs predicted')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[1].hist(y_test['calories'] - pred[:,0], bins=40)
axes[1].set_title('Calorie residuals')
plt.tight_layout()
plt.show()

## 8. Save backend bundle

In [ ]:
bundle = {
    'model': best_model,
    'features': FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'targets': TARGETS,
    'selected_model': best['model'],
    'leaderboard': leaderboard.to_dict(orient='records'),
    'mae_calories': float(best['mae_calories']),
    'mae_protein': float(best['mae_protein']),
    'r2_calories': float(best['r2_calories']),
    'r2_protein': float(best['r2_protein']),
    'activity': ['sedentary','light','moderate','active','very_active'],
    'goal': ['lose_fat','maintain','gain_muscle','recompose','improve_fitness'],
    'diet_types': ['balanced','vegetarian','vegan','high_protein','diabetes_friendly','heart_friendly'],
    'experience': ['beginner','intermediate','advanced'],
    'equipment': ['none','bands','dumbbells','gym'],
    'work_style': ['desk','mixed','standing','physical'],
    'stress': ['low','medium','high'],
    'training_notes': 'Synthetic lifestyle cohort generated from conservative guideline logic.',
    'references': [
        'CDC adult activity guidance: 150 minutes moderate activity weekly plus 2 days of muscle-strengthening - https://www.cdc.gov/physical-activity-basics/guidelines/adults.html',
        'Physical Activity Guidelines for Americans: 150-300 minutes moderate activity weekly and muscle-strengthening - https://www.cdc.gov/physical-activity/media/pdfs/Physical_Activity_Guidelines_2nd_edition.pdf',
        'CDC adult sleep guidance: at least 7 hours per 24 hours for adults - https://www.cdc.gov/sleep/about/index.html',
        'Dietary Guidelines for Americans 2020-2025: nutrient-dense foods within calorie limits - https://www.dietaryguidelines.gov/sites/default/files/2020-12/Dietary_Guidelines_for_Americans_2020-2025.pdf',
    ],
}
joblib.dump(bundle, MODELS_DIR / 'lifestyle_model.pkl')
print('Saved:', MODELS_DIR / 'lifestyle_model.pkl')

## 9. What the backend adds

The model estimates calories and protein. The Flask route then layers on the planner features:

- macro and fiber targets;
- hydration target;
- meal timing and grocery map;
- weekly training schedule;
- four-week progression calendar;
- daily routine, habit stack, and safety flags.